# Driver Drowsiness Detection

This notebook integrates the complete real-time driver drowsiness
detection system.

The system combines:

- MediaPipe Face Landmarker for facial landmark detection
- Eye landmark-based eye region extraction
- A fine-tuned CNN for eye-state classification
- Mouth Aspect Ratio (MAR) for yawn detection
- Temporal drowsiness logic
- Real-time visual feedback
- An alarm system

The final system processes webcam frames continuously and displays
the driver's eye state, prediction confidence, mouth activity,
drowsiness indicators, and final status.

## 1. Import Required Libraries

The detection and inference logic is implemented inside the
`src` modules.

This notebook acts as the integration layer and does not
reimplement those components.

In [1]:
import cv2
import time
import numpy as np
import os
import sys
from pathlib import Path

PATH = Path.cwd().parent
sys.path.append(str(PATH))


In [11]:
from src.face_detection import FaceDetector
from src.eye_detection import EyeDetector
import inspect
from src.eye_inference import EyeInference
from src.yawn_detection import calculate_mar, is_yawning
from src.drowsiness_logic import DrowsinessDetector
import winsound
print(inspect.getsource(FaceDetector.detect))

    def detect(self, frame, timestamp_ms):
        """
        Detect facial landmarks for the current frame.

        Parameters
        ----------
        frame : numpy.ndarray
            OpenCV BGR frame.

        timestamp_ms : int
            Increasing timestamp in milliseconds.

        Returns
        -------
        FaceLandmarkerResult
            MediaPipe face landmark result.
        """

        # OpenCV BGR → RGB
        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Convert to MediaPipe Image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # Synchronous detection
        result = self.landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        return result



In [12]:
cap = cv2.VideoCapture(1)

face_detector = FaceDetector()

timestamp = 0

while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read frame.")
        break

    timestamp += 33

    result = face_detector.detect(
        frame,
        timestamp
    )

    if len(result.face_landmarks) > 0:

        cv2.putText(
            frame,
            "FACE DETECTED",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    else:

        cv2.putText(
            frame,
            "NO FACE DETECTED",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

    cv2.imshow(
        "Face Detection Test",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
face_detector.close()
cv2.destroyAllWindows()

## 2. Initialize Detection Components

Each component is responsible for its own internal configuration.

- `FaceDetector` initializes MediaPipe.
- `EyeInference` loads the trained MRL model.
- `EyeDetector` extracts the eye regions.
- `DrowsinessDetector` maintains temporal state.

No duplicate model loading or MediaPipe initialization is required
in this notebook.

In [13]:
face_detector = FaceDetector()
eye_detector = EyeDetector()
eye_inference = EyeInference()
drowsiness_detector = DrowsinessDetector()

Loading model...
Model loading in successfull


## 3. Initialize Webcam

The webcam provides the continuous video stream that will be
processed frame by frame.

## 4. Real-Time Detection Pipeline

For every video frame:

1. Submit the frame to the Face Landmarker.
2. Retrieve the latest facial landmark result.
3. Extract the left and right eye regions.
4. Classify both eyes using the trained CNN.
5. Calculate the Mouth Aspect Ratio (MAR).
6. Detect a possible yawn.
7. Pass eye and yawn states to the drowsiness logic.
8. Display the results.
9. Trigger the alarm when drowsiness is detected.

The Face Landmarker operates in `LIVE_STREAM` mode, therefore
face detection is asynchronous and the latest available result
is retrieved from the detector.

In [14]:
import threading
alarm_stop_event = threading.Event()
alarm_thread = None

def alarm_loop():
    while not alarm_stop_event.is_set():
        winsound.Beep(1000,500)
        if alarm_stop_event.wait(0.5):
            break
        
def start_alarm():
    global alarm_thread
    if alarm_thread is not None and alarm_thread.is_alive():
        return
    alarm_stop_event.clear()
    alarm_thread = threading.Thread(target=alarm_loop,daemon=True)
    alarm_thread.start()

def stop_alarm():
    alarm_stop_event.set()




In [15]:
cap = cv2.VideoCapture(1)
if not cap.isOpened():
    raise RuntimeError("Could not open camera")

print("Webcam is opened successfully")

alarm_active = False

previous_status = "Awake"

frame_timestamp = 0

print("Starting real-time detection...")
print("Press 'q' to quit.")


while True:
    ret,frame = cap.read()

    if not ret:
        print("Failed to read frame.")
        break

    frame_timestamp += 33

    

    result = face_detector.detect(frame,frame_timestamp)

    if result is None or len(result.face_landmarks)==0:
        cv2.putText(
            frame,
            "NO FACE DETECTED",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

        cv2.imshow(
            "Driver Drowsiness Detection",
            frame
        )

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        continue

    face_landmark = result.face_landmarks[0]

    frame_height,frame_width = frame.shape[:2]

    eye_data = eye_detector.detect(frame,face_landmark)

    left_eye = eye_data["left_eye"]
    right_eye = eye_data["right_eye"]

    left_box = eye_data["left_box"]
    right_box = eye_data["right_box"]

    left_label,left_confidence,left_probability = eye_inference.predict(left_eye)
    right_label,right_confidence,right_probability = eye_inference.predict(right_eye)

    left_eye_closed = left_label == "Sleepy"
    right_eye_closed = right_label == "Sleepy"

    mar = calculate_mar(face_landmark,frame_width,frame_height)

    yawning = is_yawning(mar)

    drowsiness_result = drowsiness_detector.update(left_eye_closed,right_eye_closed,yawning)
    status = drowsiness_result["status"]
    lx1, lx2, ly1, ly2 = left_box
    rx1, rx2, ry1, ry2 = right_box

    cv2.rectangle(
        frame,
        (lx1, ly1),
        (lx2, ly2),
        (255, 255, 255),
        2
    )

    cv2.rectangle(
        frame,
        (rx1, ry1),
        (rx2, ry2),
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Left Eye: {left_label} "
        f"{left_confidence * 100:.1f}%",
        (30, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Right Eye: {right_label} "
        f"{right_confidence * 100:.1f}%",
        (30, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"MAR: {mar:.2f}",
        (30, 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Yawning: {'YES' if yawning else 'NO'}",
        (30, 140),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Eye Closed: "
        f"{drowsiness_result['eye_closed_duration']:.1f}s",
        (30, 180),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Yawn Duration: "
        f"{drowsiness_result['yawn_duration']:.1f}s",
        (30, 210),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"STATUS: {drowsiness_result['status'].upper()}",
        (30, 260),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 255, 0) if drowsiness_result['status'] == "Awake"
        else (0, 0, 255),
        3
    )

    if status == "Drowsy":
       start_alarm()

    elif status == "Awake":
        stop_alarm()

    cv2.imshow(
        "Driver Drowsiness Detection",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()

face_detector.close()

cv2.destroyAllWindows()

print("Drowsiness detection system stopped.")


Webcam is opened successfully
Starting real-time detection...
Press 'q' to quit.
Drowsiness detection system stopped.
